# The CARLA campaign, across towns

`carla-campaign.ipynb` demonstrates the recommendation transfers from the
synthetic map to **one** real town. This asks whether it transfers to
*towns*, plural — the same question the 25-map synthetic sweep answered for
the planar generator, and the same answer format: does the **ranking** hold
even when the absolute numbers move?

The synthetic sweep found the map dominates the seed as a variance source
(between-map sd 3.4x within-map for served) while the **ranking stayed
map-independent** — `Reordering W900` won 22 of 25 maps and was never worse
than 2nd. If CARLA behaves the same way, the recommendation is safe to state
generally rather than as a fact about Town01.

## The scenario is held identical across towns, deliberately

Same requests, same interval, same capacity, same window. Towns differ in
size, so an identical scenario will be easier in a small town and harder in
a large one — **that variation is the finding**, not a confound to be
normalised away. It is exactly what the synthetic map-seed sweep did, and
normalising per town would destroy the comparison it exists to make.

The one thing that must scale is `MAX_TIME`, which is set generously enough
for the largest town: a ceiling that truncates a run looks exactly like a
coverage failure.

## Every town needs three artifacts first

| artifact | how | automated? |
|---|---|---|
| `<Town>.topology.json` | `src.map.carla_dump` | yes |
| `<Town>.stops.json` | `notebooks/carla-stop-authoring.ipynb` | **no — hand-authored** |
| `traffic_efficiency` | `src.map.carla_calibrate` | yes |

Only **Town01** has all three today. Section 2 reports what is missing per
town; sections 1 and 3 produce the automated parts.

**`traffic_efficiency` does not transfer between towns.** It is dominated by
red-light density, which is a property of the map. Town01's 0.455 means
nothing anywhere else, and using it would repeat the 1.75x modelling error
the old CARLA runs were purged for.

## ⚠ Placeholders — update these before trusting anything

This notebook was written without a live server, so three things are
guesses. Each is marked `# ⚠ PLACEHOLDER` in the code.

| what | where | how to resolve |
|---|---|---|
| **Which towns** | `TOWNS` in the config | Run **section 1** against a live server; it surveys every installed town and reports the metrics that should drive the choice. |
| **Expected minutes per trial** | `EXPECTED_TRIAL_MINUTES` | Only feeds the ETA. Update once a few trials have run — bigger towns have longer legs and will be slower than Town01's ~3 min. |
| **Per-town calibration reference** | `CALIBRATION_REFERENCE` | Filled in automatically by **section 3**; the check in section 7 needs it to verify the calibration holds when the policy picks the legs. |

The town list below is a *reasoned* placeholder, not a random one — it aims
for variety in road type rather than coverage of everything:

- **Town03** — the most varied built-in: roundabout, tunnel, elevation.
- **Town05** — multi-lane grid, wider roads than Town01.
- **Town07** — rural, narrow, few traffic lights, which should push
  `traffic_efficiency` toward 1.0 and test whether the calibration recipe
  behaves across a real range.

**Town10HD_Opt is deliberately excluded**: it was already measured as barely
able to move a bus even on plain autopilot.

## Configuration

In [ ]:
# ⚠ PLACEHOLDER -- confirm against section 1's survey on a live server.
# Chosen for variety of road type rather than for coverage: Town03 is the
# most varied built-in (roundabout, tunnel, elevation), Town05 is a wider
# multi-lane grid, Town07 is rural with few signals. Town10HD_Opt is
# excluded on measured grounds -- it barely moves a bus on plain autopilot.
TOWNS = ["Town03", "Town05", "Town07"]

# Town01 is the reference the others are read against; it is not re-run here.
REFERENCE_TOWN = "Town01"

# ⚠ PLACEHOLDER -- ETA only, currently Town01's measured ~3 min. Bigger towns
# have longer legs and will be slower. Update once a few trials have run.
EXPECTED_TRIAL_MINUTES = {"Town03": 4.0, "Town05": 4.0, "Town07": 3.5}
DEFAULT_TRIAL_MINUTES = 4.0

# ⚠ PLACEHOLDER -- written by section 3's calibration. Until then the
# calibration check in section 7 has nothing to compare against.
# Shape: {town: {"speed_mps": float, "red_fraction": float}}
CALIBRATION_REFERENCE = {
    REFERENCE_TOWN: {"speed_mps": 4.21, "red_fraction": 0.321},
}

# --- Identical to carla-campaign.ipynb, on purpose ---------------------
NPC_VEHICLES = 20
SEEDS = list(range(10))
MAX_REQUESTS = 12
DEMAND_INTERVAL_S = 150.0
CAPACITY = 20
WINDOW = 900.0            # what the policies plan against
PATIENCE = 1800.0         # looser than the window, so a breach is observable
DWELL_S = 10.0
FIXED_DELTA_S = 0.05

# Raised above the single-town notebook's 10,000: the same scenario in a
# larger town takes longer, and a ceiling that truncates a run is
# indistinguishable from a coverage failure.
MAX_TIME = 20000.0

CARLA_HOST, CARLA_PORT = "localhost", 2000

In [ ]:
import math
import statistics
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from cycler import cycler
from scipy import stats
from tqdm.auto import tqdm

from src.algorithm.FixedRoutePolicy import FixedRoutePolicy
from src.algorithm.ReorderingInsertionPolicy import ReorderingInsertionPolicy
from src.algorithm.WindowInsertionPolicy import WindowInsertionPolicy
from src.experiment import run_campaign
from src.experiment.store import DEFAULT_RUNS_DIR, load_runs, to_results
from src.map.carla_stops import available_towns, stops_path
from src.map.carla_topology import topology_path
from src.map.CarlaMapProvider import CarlaMapProvider
from src.simulation.CarlaBackend import CarlaBackend, stop_positions_from

from IPython.display import Markdown, display

# Validated palette slots 1-3 (all-pairs CVD 9.2, normal-vision 27.6 on this
# surface). Every chart ships with its table, which is the relief the green
# slot's sub-3:1 contrast requires.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.prop_cycle": cycler(color=SERIES),
    "axes.edgecolor": INK_SOFT, "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "grid.linewidth": 0.8, "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110,
})

POLICY_LABELS = ["Fixed route", "Window W900", "Reordering W900"]
POLICIES = [
    ("Fixed route", lambda g: FixedRoutePolicy(g)),
    ("Window W900", lambda g: WindowInsertionPolicy(g, max_wait_s=WINDOW)),
    ("Reordering W900",
     lambda g: ReorderingInsertionPolicy(g, max_wait_s=WINDOW)),
]

ALL_TOWNS = [REFERENCE_TOWN] + TOWNS
TRIAL = dict(
    max_requests=MAX_REQUESTS, demand_interval_s=DEMAND_INTERVAL_S,
    capacity=CAPACITY, max_wait_s=PATIENCE, dwell_s=DWELL_S,
    max_time=MAX_TIME,
)
print(f"{len(TOWNS)} towns x {len(POLICIES)} policies x {len(SEEDS)} seeds "
      f"= {len(TOWNS) * len(POLICIES) * len(SEEDS)} trials")

## 1. Survey the towns  🖥 *needs a live server*

Resolves the `TOWNS` placeholder with evidence instead of memory. Loads each
installed town in turn and measures what should actually drive the choice:
how much road there is, how much of it is junction, how varied the speed
limits are, and — the one that decides whether a town is usable at all —
whether it has **Shoulder** lanes, since every authored stop needs one to
its right and passengers are spawned there.

Loading each town takes a few seconds, so this is a couple of minutes.

In [ ]:
def survey_town(client, town: str) -> dict:
    """
    Measure one town's road network on a live server.
    :param client: Connected carla.Client.
    :param town: Town name to load and measure.
    :return: One row of survey metrics.
    """
    import carla

    world = client.load_world(town)
    carla_map = world.get_map()
    topology = carla_map.get_topology()

    length = junction_length = 0.0
    for entry, exit_waypoint in topology:
        span = entry.transform.location.distance(
            exit_waypoint.transform.location)
        length += span
        if entry.is_junction:
            junction_length += span

    # Shoulders decide whether a stop can be authored here at all.
    waypoints = carla_map.generate_waypoints(5.0)
    with_shoulder = sum(
        1 for w in waypoints
        if w.lane_type == carla.LaneType.Driving
        and (right := w.get_right_lane()) is not None
        and right.lane_type == carla.LaneType.Shoulder
    )
    driving = sum(1 for w in waypoints
                  if w.lane_type == carla.LaneType.Driving)
    landmarks = carla_map.get_all_landmarks_of_type("274")
    lights = world.get_actors().filter("traffic.traffic_light*")

    return {
        "town": town,
        "road_km": round(length / 1000, 2),
        "junction_%": round(100 * junction_length / length, 1) if length else 0,
        "driving_wps": driving,
        "shoulder_%": round(100 * with_shoulder / driving, 1) if driving else 0,
        "speed_signs": len(landmarks),
        "speed_values": sorted({float(l.value) for l in landmarks}),
        "traffic_lights": len(lights),
    }


def discover_towns(client) -> list[str]:
    """
    List the towns this server has, excluding layered duplicates.
    :param client: Connected carla.Client.
    :return: Town names, deduplicated and sorted.
    """
    names = sorted({
        name.split("/")[-1] for name in client.get_available_maps()
    })
    # The *_Opt variants are the same towns with toggleable layers; surveying
    # both doubles the work and tells us nothing new.
    return [n for n in names if not n.endswith("_Opt")]


RUN_SURVEY = True   # set False to skip; needs a live server

if RUN_SURVEY:
    import carla
    client = carla.Client(CARLA_HOST, CARLA_PORT)
    client.set_timeout(120.0)
    candidates = discover_towns(client)
    print(f"server has {len(candidates)} towns: {candidates}")

    rows = []
    for town in tqdm(candidates, desc="surveying towns", unit="town"):
        try:
            rows.append(survey_town(client, town))
        except Exception as error:              # noqa: BLE001
            print(f"  {town}: FAILED {error!r}")
    survey = pd.DataFrame(rows).set_index("town")
    display(survey)
    display(Markdown(
        "**Reading this.** `shoulder_%` near zero means stops cannot be "
        "authored there without relaxing the legality rule — treat it as "
        "disqualifying. Prefer variety in `road_km`, `junction_%` and "
        "`speed_values` over picking the biggest; the point is a *range* of "
        "towns, and Town01 already covers small-grid-urban."))
else:
    survey = None
    print("survey skipped")

## 2. What is prepared, and what is missing  *(no server needed)*

Run this any time. Stop authoring is the manual step and cannot be automated
from here — it needs `notebooks/carla-stop-authoring.ipynb` and a look at
the map.

In [ ]:
def readiness(town: str) -> dict:
    """
    Report which of a town's three artifacts exist.
    :param town: Town name.
    :return: A row describing what is present and what is not.
    """
    has_topology = topology_path(town).exists()
    has_stops = stops_path(town).exists()
    efficiency, stops, note = None, None, ""
    if has_topology and has_stops:
        try:
            provider = CarlaMapProvider(town)
            efficiency = provider.traffic_efficiency()
            stops = len(provider.get_stops())
            problems = provider.validate()
            note = "OK" if not problems else f"{len(problems)} map problems"
        except Exception as error:              # noqa: BLE001
            note = f"failed to load: {error!r}"
    return {
        "town": town,
        "topology": "yes" if has_topology else "MISSING",
        "stops": stops if stops is not None else
                 ("yes" if has_stops else "MISSING"),
        "traffic_efficiency": efficiency if efficiency is not None else "—",
        "calibrated": ("yes" if efficiency not in (None, 1.0)
                       else "NO" if efficiency == 1.0 else "—"),
        "note": note,
    }


status = pd.DataFrame([readiness(t) for t in ALL_TOWNS]).set_index("town")
display(status)

needs_stops = [t for t in ALL_TOWNS if not stops_path(t).exists()]
needs_calibration = [
    t for t in ALL_TOWNS
    if topology_path(t).exists() and stops_path(t).exists()
    and CarlaMapProvider(t).traffic_efficiency() == 1.0
]
if needs_stops:
    display(Markdown(
        f"> **Hand-authoring needed for: {', '.join(needs_stops)}.** Run "
        "`notebooks/carla-stop-authoring.ipynb` once per town. This is the "
        "one step nothing here can do for you, and the campaign cannot run "
        "without it."))
if needs_calibration:
    display(Markdown(
        f"> **Uncalibrated: {', '.join(needs_calibration)}.** Section 3 "
        "fixes this. A `traffic_efficiency` of 1.0 assumes the bus achieves "
        "the posted limit exactly, which measured **1.75x optimistic** on "
        "Town01."))
if not needs_stops and not needs_calibration:
    display(Markdown("> **All towns ready.** Skip to section 4."))

## 3. Prepare the towns  🖥 *needs a live server*

Dumps the road network and calibrates what a bus achieves on it, per town.
Skips towns that have no authored stops, since calibration drives between
them.

Budget roughly 10–15 minutes per town: the dump is quick, the calibration
drives three full tours of every stop.

In [ ]:
from src.map.carla_calibrate import calibrate
from src.map.carla_dump import dump_world
from src.map.carla_topology import CarlaTopology, load_topology, save_topology

CALIBRATION_TOURS = 3

RUN_PREPARE = True   # set False to skip; needs a live server

if RUN_PREPARE:
    import carla
    client = carla.Client(CARLA_HOST, CARLA_PORT)
    client.set_timeout(120.0)

    todo = [t for t in ALL_TOWNS if stops_path(t).exists()]
    print(f"preparing {todo}; skipping {set(ALL_TOWNS) - set(todo)} "
          f"(no authored stops)")

    for town in tqdm(todo, desc="preparing towns", unit="town"):
        bar = tqdm(total=2, desc=f"  {town}", leave=False, unit="step")

        # 1. Dump the roads, reading per-road speed limits from the signs.
        bar.set_postfix_str("dumping roads")
        world = client.load_world(town)
        topology = dump_world(world)
        save_topology(topology, note=(
            f"dumped for the cross-town campaign; "
            f"{len(topology.speed_limit_overrides)} signed roads"))
        bar.update(1)

        # 2. Calibrate, unless it already carries a factor. Re-calibrating
        #    compounds, since the drive happens under the factor in force.
        current = load_topology(town).traffic_efficiency
        if current != 1.0:
            bar.set_postfix_str(f"already calibrated ({current})")
            bar.update(1)
            bar.close()
            continue

        bar.set_postfix_str(f"calibrating ({CALIBRATION_TOURS} tours)")
        summary = calibrate(town, CALIBRATION_TOURS, NPC_VEHICLES,
                            CARLA_HOST, CARLA_PORT, verbose=False)
        save_topology(
            CarlaTopology(
                map=topology.map,
                default_speed_limit_kph=topology.default_speed_limit_kph,
                speed_limit_overrides=topology.speed_limit_overrides,
                segments=topology.segments,
                traffic_efficiency=summary["traffic_efficiency"],
            ),
            note=(f"traffic_efficiency {summary['traffic_efficiency']} over "
                  f"{summary['legs_arrived']} legs at {NPC_VEHICLES} NPCs"))
        CALIBRATION_REFERENCE[town] = {
            "speed_mps": summary["effective_speed_mps"],
            "red_fraction": summary["red_light_fraction"],
        }
        bar.update(1)
        bar.close()
        print(f"  {town}: efficiency {summary['traffic_efficiency']}, "
              f"bias {summary['bias']}x, "
              f"sigma within {summary['sigma_within_leg']}, "
              f"between {summary['sigma_between_legs']}, "
              f"red {100 * summary['red_light_fraction']:.0f}%")

    display(Markdown(
        "**Copy `CALIBRATION_REFERENCE` into the config cell** so section 7 "
        "still has it after a kernel restart:\n\n```python\n"
        f"CALIBRATION_REFERENCE = {CALIBRATION_REFERENCE}\n```"))
else:
    print("preparation skipped")

## 4. Run the campaign  🖥 *needs a live server*

Sequential — one server cannot host a campaign per core. Per seed rather
than per campaign, so a CARLA hiccup costs one trial instead of the night,
and everything already recorded is skipped so re-running resumes.

Two nested progress bars: towns outside, seeds inside, with a rolling
per-trial time so the ETA reflects what this town is actually costing rather
than what Town01 cost.

In [ ]:
def world_for(town: str, provider: CarlaMapProvider) -> dict:
    """
    Build the store identity for one town's world.

    traffic_efficiency and the speed-limit rule are in here because they
    change every estimate the policy makes; omitting them would let a
    pre-calibration run answer for a post-calibration one.
    :param town: Town name.
    :param provider: That town's loaded provider.
    :return: The world identity.
    """
    return {
        "town": town, "backend": "CarlaBackend",
        "npc_vehicles": NPC_VEHICLES, "fixed_delta_s": FIXED_DELTA_S,
        "no_rendering": True, "blueprint": "vehicle.volkswagen.t2_2021",
        "traffic_efficiency": provider.traffic_efficiency(),
        "speed_limits": "landmark-harmonic-v1",
    }


def backend_factory_for(town: str, positions: dict):
    """
    Build a per-seed world builder for one town.
    :param town: Town to drive in.
    :param positions: That town's stop coordinates.
    :return: A function of seed returning a backend factory.
    """
    def for_seed(seed: int):
        def build(graph, start_stop_id: str) -> CarlaBackend:
            """Build the CARLA backend for one trial."""
            return CarlaBackend(
                stop_positions=positions, position=start_stop_id, town=town,
                host=CARLA_HOST, port=CARLA_PORT,
                fixed_delta_s=FIXED_DELTA_S, dwell_s=DWELL_S,
                npc_vehicles=NPC_VEHICLES, seed=seed, no_rendering=True,
            )
        return build
    return for_seed


RUN_CAMPAIGN = True   # set False to skip; needs a live server

if RUN_CAMPAIGN:
    ready = [t for t in TOWNS if stops_path(t).exists()]
    if not ready:
        display(Markdown(
            "> **Nothing to run.** No town in `TOWNS` has authored stops "
            "yet — see section 2."))

    failures, durations = [], []
    outer = tqdm(ready, desc="towns", unit="town")
    for town in outer:
        outer.set_postfix_str(town)
        provider = CarlaMapProvider(town)
        positions = stop_positions_from(provider)
        if provider.traffic_efficiency() == 1.0:
            print(f"  {town}: SKIPPED — uncalibrated, see section 3")
            continue

        expected = EXPECTED_TRIAL_MINUTES.get(town, DEFAULT_TRIAL_MINUTES)
        inner = tqdm(SEEDS, desc=f"  {town} seeds", unit="seed", leave=False)
        for seed in inner:
            started = time.perf_counter()
            try:
                run_campaign(
                    policies=POLICIES, seeds=[seed], world=world_for(town, provider),
                    directory=DEFAULT_RUNS_DIR, provider=provider,
                    backend_factory_for=backend_factory_for(town, positions),
                    n_jobs=1, progress=False, **TRIAL)
            except Exception as error:          # noqa: BLE001
                # Broad on purpose: a CARLA failure surfaces as almost
                # anything, and losing the remaining seeds to it is worse.
                failures.append((town, seed, repr(error)))
                inner.write(f"  {town} seed {seed} FAILED: {error!r}")
            elapsed = (time.perf_counter() - started) / 60
            durations.append(elapsed)
            rolling = statistics.fmean(durations[-5:])
            inner.set_postfix_str(
                f"{elapsed:.1f} min/seed, rolling {rolling:.1f} "
                f"(expected {expected * len(POLICIES):.1f})")
        inner.close()

    print(f"\ndone — {len(durations)} seeds attempted, "
          f"{len(failures)} failed")
    for town, seed, error in failures:
        print(f"  {town} seed {seed}: {error}")
else:
    print("campaign skipped")

## 5. Collect

In [ ]:
records = [
    r for r in load_runs(DEFAULT_RUNS_DIR)
    if r.identity.world.get("backend") == "CarlaBackend"
    and r.identity.world.get("town") in ALL_TOWNS
    and r.identity.world.get("npc_vehicles") == NPC_VEHICLES
    and r.identity.world.get("speed_limits") == "landmark-harmonic-v1"
    and r.identity.trial.get("max_requests") == MAX_REQUESTS
    and r.identity.policy in POLICY_LABELS
]

frame = pd.DataFrame([{
    "town": r.identity.world["town"], "policy": r.identity.policy,
    "seed": r.identity.seed,
    "served": r.kpi.served_fraction * 100,
    "wait": r.kpi.mean_wait_time, "ride": r.kpi.mean_ride_time,
    "throughput": r.kpi.throughput,
    "breach": (100.0 * sum(w > WINDOW for w in r.wait_times)
               / len(r.wait_times) if r.wait_times else float("nan")),
    "sim_s": r.backend_stats.get("ticks", 0) * FIXED_DELTA_S,
    "distance_m": r.backend_stats.get("distance_m", 0.0),
    "red_s": r.backend_stats.get("red_light_ticks", 0) * FIXED_DELTA_S,
    "repaths": r.backend_stats.get("repaths", 0),
    "unroutable": r.backend_stats.get("unroutable_legs", 0),
} for r in records], columns=[
    "town", "policy", "seed", "served", "wait", "ride", "throughput",
    "breach", "sim_s", "distance_m", "red_s", "repaths", "unroutable"])

HAVE_RUNS = not frame.empty
print(f"{len(frame)} runs across {frame['town'].nunique() if HAVE_RUNS else 0}"
      f" towns")
if HAVE_RUNS:
    display(frame.groupby(["town", "policy"])[["served", "wait", "ride"]]
            .mean().round(1))
    suspect = frame[(frame["repaths"] > 0) | (frame["unroutable"] > 0)]
    print(f"runs with repaths or unroutable legs: {len(suspect)}"
          + ("" if suspect.empty else "  <- inspect before trusting these"))
else:
    display(Markdown(
        "> **No runs yet.** Run section 4, or check that the world identity "
        "here matches what it filed under."))

## 6. Does the ranking hold across towns?

**The question this notebook exists to answer.** The synthetic sweep found
the map moves the absolute numbers a great deal while leaving the *ranking*
alone. If CARLA agrees, the recommendation is a general claim rather than a
fact about Town01.

In [ ]:
if not HAVE_RUNS:
    display(Markdown("_Nothing to report until the campaign has run._"))
else:
    means = frame.groupby(["town", "policy"])["served"].mean().unstack()
    ranked = means.rank(axis=1, ascending=False).astype(int)

    figure, axes = plt.subplots(1, 2, figsize=(12.5, 4.0))
    towns = list(means.index)
    width = 0.26
    for offset, (colour, label) in enumerate(zip(SERIES, POLICY_LABELS)):
        if label not in means:
            continue
        axes[0].bar([i + (offset - 1) * width for i in range(len(towns))],
                    means[label], width * 0.92, color=colour, label=label)
    axes[0].axhline(80, ls="--", lw=1.1, color=INK_SOFT)
    axes[0].set_xticks(range(len(towns)), towns)
    axes[0].set_ylabel("served (%)")
    axes[0].set_title("Coverage per town (dashed = the 80% gate)")
    axes[0].grid(axis="y")
    axes[0].set_axisbelow(True)
    axes[0].legend()

    for colour, label in zip(SERIES, POLICY_LABELS):
        if label in ranked:
            axes[1].plot(range(len(towns)), ranked[label], marker="o",
                         markersize=8, lw=2, color=colour, label=label)
    axes[1].set_xticks(range(len(towns)), towns)
    axes[1].set_yticks([1, 2, 3])
    axes[1].invert_yaxis()
    axes[1].set_ylabel("rank by coverage")
    axes[1].set_title("Ranking — flat lines mean the ranking transfers")
    axes[1].grid(axis="y")
    axes[1].set_axisbelow(True)
    axes[1].legend()
    figure.tight_layout()
    plt.show()

    display(means.round(1))
    winners = means.idxmax(axis=1)
    display(Markdown(
        "**Won by:** "
        + ", ".join(f"{t} — {w}" for t, w in winners.items())
        + f"\n\nSynthetic reference: `Reordering W900` won **22 of 25** maps "
        "and was never worse than 2nd; only the top two ever swapped."))

## 7. §3.2.2, the L1 advantage, and the calibration — per town

The same three checks the single-town notebook makes, run once per town so
the answers can be compared rather than pooled. Pooling would hide exactly
the between-town variation this notebook is looking for.

In [ ]:
if not HAVE_RUNS:
    display(Markdown("_Nothing to report until the campaign has run._"))
else:
    lines = ["| town | policy | fulfilment | sum of % diffs | gate | §3.2.2 |",
             "|---|---|---|---|---|---|"]
    for town in sorted(frame["town"].unique()):
        block = frame[frame["town"] == town]
        town_means = block.groupby("policy")[
            ["served", "wait", "ride", "throughput"]].mean()
        if "Fixed route" not in town_means.index:
            continue
        base = town_means.loc["Fixed route"]
        for label in POLICY_LABELS[1:]:
            if label not in town_means.index:
                continue
            row = town_means.loc[label]
            total = (100 * (base["wait"] - row["wait"]) / base["wait"]
                     + 100 * (base["ride"] - row["ride"]) / base["ride"]
                     + 100 * (row["throughput"] - base["throughput"])
                     / base["throughput"])
            gate = row["served"] >= 80.0
            lines.append(
                f"| {town} | {label} | {row['served']:.1f}% | {total:+.1f} "
                f"| {'✓' if gate else '✗'} | "
                f"{'**PASS**' if gate and total > 0 else 'FAIL'} |")
    display(Markdown("\n".join(lines)))

In [ ]:
if not HAVE_RUNS:
    display(Markdown("_Nothing to report until the campaign has run._"))
else:
    lines = ["| town | L0 ride | L1 ride | L1 − L0 | p | seeds |",
             "|---|---|---|---|---|---|"]
    for town in sorted(frame["town"].unique()):
        block = frame[frame["town"] == town]
        l0 = block[block["policy"] == "Window W900"].set_index("seed")
        l1 = block[block["policy"] == "Reordering W900"].set_index("seed")
        shared = sorted(set(l0.index) & set(l1.index))
        if len(shared) < 2:
            continue
        a = l0.loc[shared, "ride"].astype(float)
        b = l1.loc[shared, "ride"].astype(float)
        p = (stats.ttest_rel(b, a).pvalue if (b - a).std() > 0
             else float("nan"))
        lines.append(f"| {town} | {a.mean():.0f} s | {b.mean():.0f} s | "
                     f"**{(b - a).mean():+.0f} s** | {p:.4f} | {len(shared)} |")
    display(Markdown("\n".join(lines)))
    display(Markdown(
        "Synthetic reference: **−111.5 s** pooled over 25 maps, **−70 s** on "
        "the campus map. At ten seeds a town treat the sign and rough "
        "magnitude as the claim, not the p-value."))

In [ ]:
if not HAVE_RUNS:
    display(Markdown("_Nothing to report until the campaign has run._"))
else:
    drive = frame[frame["sim_s"] > 0].copy()
    drive["speed"] = drive["distance_m"] / drive["sim_s"]
    drive["red_frac"] = drive["red_s"] / drive["sim_s"]
    observed = drive.groupby("town")[["speed", "red_frac"]].mean()

    lines = ["| town | campaign m/s | calibration m/s | Δ | campaign red % "
             "| calibration red % |", "|---|---|---|---|---|---|"]
    for town, row in observed.iterrows():
        reference = CALIBRATION_REFERENCE.get(town)
        if reference is None:
            lines.append(f"| {town} | {row['speed']:.2f} | ⚠ not recorded "
                         f"| — | {100 * row['red_frac']:.1f}% | — |")
            continue
        delta = 100 * (row["speed"] / reference["speed_mps"] - 1)
        lines.append(
            f"| {town} | {row['speed']:.2f} | {reference['speed_mps']:.2f} "
            f"| {delta:+.1f}% | {100 * row['red_frac']:.1f}% "
            f"| {100 * reference['red_fraction']:.1f}% |")
    display(Markdown("\n".join(lines)))
    display(Markdown(
        "Within a few percent means `traffic_efficiency` describes the town "
        "rather than the tour it was measured on — which is what makes the "
        "per-town calibration recipe trustworthy for the maps still to "
        "come. A `⚠ not recorded` row means section 3's "
        "`CALIBRATION_REFERENCE` was never copied into the config."))

## 8. What to conclude

Fill in once run.

1. **Does the ranking transfer?** The headline. Flat lines in section 6's
   right-hand panel mean the recommendation is a general claim; crossing
   lines mean it is a fact about particular towns, and the writeup has to
   say which.
2. **How much do the absolute numbers move?** The synthetic sweep found
   between-map sd 3.4x within-map for coverage. If CARLA is comparable, that
   is independent support for the "quote two numbers, clearly labelled"
   discipline — one for the town, one across towns.
3. **Does §3.2.2 hold everywhere?** It passes on 21 of 25 synthetic maps,
   with every failure at the fulfilment gate and never the sum. If a town
   fails here, check whether it is load — a bigger town with the same demand
   is a harder scenario, and the load sweep showed those failures recover as
   demand thins.
4. **Does the calibration recipe hold up?** Section 7's last table is the
   one that matters for everything after this notebook: if `traffic_efficiency`
   describes each town as well as it described Town01, then adding a map —
   built-in or `.osm` — is a mechanical procedure rather than a research
   question.

Also record: any town that needed repaths or hit unroutable legs, and any
town excluded during the survey and why. A town that could not be used is
a finding about the pipeline's reach, not an empty row.